# 🔎 01. Preprocessing
## 🔎 Metody mieszane w analizie tekstu: od słowników do BERT

**Cel:** zobaczyć, jak decyzje o tekście zmieniają pomiar. Dokument = jeden wiersz; token = jednostka podziału; słownik korpusu = zbiór różnych tokenów. Angielski korpus zachowujemy w oryginalnym języku. „Surowy” oznacza tu tekst dostarczony w pliku już nazwanym cleaned, nie pierwotny eksport Reddita.

# 🧭 Jak wykonać ten notebook

1. Otwórz plik `.ipynb` w Google Colab i zapisz własną kopię na Dysku Google.
2. **Wystarczy CPU.** Komórka to jeden blok tekstu albo kodu. Kod uruchamiasz przyciskiem ▶ po lewej lub Shift+Enter.
3. Uruchom pierwszą komórkę kodu. Jeżeli zainstaluje biblioteki i pokaże 🔄, uruchom ponownie sesję z menu **Środowisko wykonawcze**. Potem zacznij od pierwszej komórki. Restart zachowuje pliki, ale usuwa zmienne z pamięci.
4. Wykonuj kod **od góry, bez pomijania komórek**. Dane pobiorą się automatycznie z [repozytorium prowadzącego](https://github.com/bartlomiejnowak-ux/PSPS-2026). Domyślnie niczego nie wgrywasz. W razie awarii pobierania komórka podaje instrukcję ręcznego wgrania.
5. Poczekaj, aż obracający się znacznik przy komórce zniknie. Pierwsze pobranie modelu i obliczenia mogą potrwać kilka minut, a BERTopic dłużej. Nie klikaj wielokrotnie ▶.
6. ✅ oznacza sukces, 🔎 wskazuje co przeczytać, ⚠️ ważne ograniczenie, ✏️ ćwiczenie. Emoji nie zmieniają działania kodu.
7. Na końcu pobierz ZIP wyników. Sam zapis notebooka na Dysku nie zachowuje plików z tymczasowej sesji.


### 🛠️ Gdy coś nie działa

| Objaw | Co zrobić |
|---|---|
| `NameError` lub „nie zdefiniowano” | Pominięto wcześniejszy krok albo zrestartowano sesję. Wykonaj kod od początku. |
| `ModuleNotFoundError` / błąd wersji biblioteki | Uruchom instalację, zrestartuj sesję i wykonaj komórki od góry. |
| Brak pliku / zła kolumna | Ponów komórkę pobierania danych. Awaryjnie wybierz `cleaned_topic_modeling_dataset(1).csv` z repozytorium. |
| Błąd pobierania modelu | Sprawdź połączenie, zaczekaj i ponów komórkę pobierania. Nie zmieniaj nazwy modelu. |
| Błąd po zmianie parametru | Cofnij zmianę lub przywróć wartości pokazane w komentarzach i wykonaj dalsze komórki kolejno. |
| Sesja wygasła | Połącz ponownie, uruchom notebook od początku i ponownie wykonaj komórkę pobierania danych. |

Nie przechodź dalej po czerwonym błędzie. Czytaj ostatnią linijkę komunikatu. W tej kopii wyniki pojawią się dopiero po uruchomieniu kodu. Liczby na slajdach pochodzą ze sprawdzonego wcześniejszego wykonania; przy innych ustawieniach lub środowisku wynik może się różnić.

## 📖 Słowa i skróty używane w tym module

- **Preprocessing:** przygotowanie tekstu do konkretnej analizy.
- **Token:** jednostka tekstu, np. słowo; tokenizer to reguła podziału.
- **Korpus:** zbiór analizowanych tekstów.
- **Stopwords:** częste słowa, np. the; usunięcie not może zmienić sens.
- **Lemma / stem:** forma podstawowa / skrót uzyskany przez obcinanie końcówek.
- **N-gram:** ciąg n sąsiednich tokenów; bigram ma dwa.
- **JSON:** zapis listy jako tekstu, który kolejny notebook odczyta.
- **Ramka danych:** tabela z wierszami i kolumnami w Pythonie.
- **Transformer / BERT:** rodzina modeli języka, które uwzględniają kontekst słów.
- **Bag-of-words:** opis tekstu przez liczby wystąpień słów, bez ich kolejności.
- **Skala logarytmiczna / log:** ściska duże wartości, żeby długi ogon rozkładu mieścił się na wykresie. Równe odstępy na osi nie oznaczają równych przyrostów liczby słów.

# 🔎 1. Przygotowanie
Uruchom komórkę instalacji poniżej. W Colab wszystko wykonujesz w przeglądarce.

### ▶️ Krok kodu 1

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** ✅ Biblioteki gotowe albo instrukcja jednorazowego restartu 🔄.

In [ ]:
import sys, subprocess, importlib.util, importlib.metadata as metadata
from pathlib import Path
IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
PACKAGES = ['numpy==2.5.3', 'pandas==3.0.5', 'matplotlib==3.11.2', 'spacy==3.8.16', 'nltk==3.10.3', 'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl']
if sys.version_info < (3, 12):
    raise RuntimeError('⛔ Ten zestaw wersji wymaga Python 3.12 lub nowszego. Wybierz zgodne środowisko.')
def installed(spec):
    name, expected = ('en-core-web-sm', '3.8.0') if spec.startswith('https:') else spec.split('==')
    try: return metadata.version(name) == expected
    except metadata.PackageNotFoundError: return False
needed = [spec for spec in PACKAGES if not installed(spec)]
if needed and IN_COLAB:
    print('⏳ Instalacja bibliotek. Poczekaj na zakończenie tej komórki.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
    raise RuntimeError('🔄 Instalacja zakończona. Uruchom ponownie sesję z menu Środowisko wykonawcze, a następnie wykonaj komórki od początku. To jednorazowy krok po instalacji.')
if needed:
    raise RuntimeError('⛔ Brakuje wymaganych wersji. Lokalnie użyj pliku requirements właściwego modułu. W Colab komórka instaluje je sama. Braki: ' + ', '.join(needed))
print('✅ Biblioteki gotowe. Możesz uruchomić następną komórkę.')


### ▶️ Krok kodu 2

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
MODULE = "01_PREPROCESSING"
from pathlib import Path
import json, re, time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
START = time.perf_counter()
HERE = Path.cwd()
ROOT = HERE.parent if HERE.name.startswith(("01_", "02_", "03_", "04_")) else HERE
(ROOT / "data").mkdir(exist_ok=True)
OUT = ROOT / MODULE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
TARGET = {0:"Stress", 1:"Depression", 2:"Bipolar", 3:"Personality Disorder", 4:"Anxiety"}
RNG = np.random.default_rng(42)
plt.rcParams.update({"figure.dpi":120, "axes.spines.top":False, "axes.spines.right":False})
print("Folder wyników:", OUT)

# 🔎 2. Wczytanie danych
Kategorie źródłowe opisują grupy tekstów; nie potwierdzają rozpoznania klinicznego autora.

### ▶️ Krok kodu 3

Uruchom raz i poczekaj. **Oczekiwany efekt:** automatyczne pobranie danych i komunikat ✅. Przy awarii ustaw w kodzie `DATA_SOURCE = 'upload'` i wybierz **`cleaned_topic_modeling_dataset(1).csv`** z repozytorium prowadzącego.

In [ ]:
# 📥 Dane z repozytorium prowadzącego. Domyślnie niczego nie wgrywasz ręcznie.
import io, urllib.request, hashlib
DATA_SOURCE = 'github'  # awaryjnie zmień na 'upload' i uruchom komórkę ponownie
GITHUB_COMMIT = '1cb710169713a4f8ea5a0d839be6d6e1df8ab078'
GITHUB_URL = f'https://raw.githubusercontent.com/bartlomiejnowak-ux/PSPS-2026/{GITHUB_COMMIT}/cleaned_topic_modeling_dataset%281%29.csv'
EXPECTED_SHA256 = '4fb7bdc1779127e875ce3ea7fd571f85885b700937b9439bd606b7c2d67d7972'
INPUT_NAME = 'cleaned_topic_modeling_dataset(1).csv'
if IN_COLAB:
    if DATA_SOURCE == 'github':
        print('⏳ Pobieranie danych z GitHub…')
        try:
            with urllib.request.urlopen(GITHUB_URL, timeout=60) as response:
                data_bytes = response.read()
        except Exception as exc:
            raise RuntimeError('⛔ Nie udało się pobrać danych. Sprawdź internet i ponów tę komórkę. Awaryjnie ustaw DATA_SOURCE = "upload" i wybierz plik z repozytorium prowadzącego.') from exc
        if hashlib.sha256(data_bytes).hexdigest() != EXPECTED_SHA256:
            raise ValueError('⛔ Pobrany plik nie zgadza się ze sprawdzoną wersją. Nie kontynuuj analizy na tym pliku.')
    elif DATA_SOURCE == 'upload':
        from google.colab import files
        print('📂 Wybierz cleaned_topic_modeling_dataset(1).csv z repozytorium prowadzącego.')
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('⛔ Wybierz dokładnie jeden plik i ponów tę komórkę.')
        data_bytes = next(iter(uploaded.values()))
    else:
        raise ValueError('⛔ DATA_SOURCE musi mieć wartość "github" albo "upload".')
    try:
        source_df = pd.read_csv(io.BytesIO(data_bytes), keep_default_na=False)
    except Exception as exc:
        raise ValueError('⛔ Nie można odczytać pliku. Wgraj cleaned_topic_modeling_dataset(1).csv z repozytorium prowadzącego.') from exc
    missing_columns = {'target', 'document'} - set(source_df.columns)
    if source_df.empty or missing_columns:
        raise ValueError(f'⛔ Pusty lub niewłaściwy plik. Brakujące kolumny: {sorted(missing_columns)}')
    data_folder = ROOT / 'data'
    data_folder.mkdir(parents=True, exist_ok=True)
    (data_folder / INPUT_NAME).write_bytes(data_bytes)
    print(f'✅ Dane gotowe: {len(source_df)} wierszy. Źródło: {DATA_SOURCE}.')


### ▶️ Krok kodu 4

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
df = pd.read_csv(ROOT / "data" / "cleaned_topic_modeling_dataset(1).csv")
assert {"target", "document"}.issubset(df.columns), "Potrzebne kolumny: target, document."
assert df.target.isin(TARGET).all(), "Nieznane lub brakujące etykiety target."
df.insert(0, "source_row", np.arange(len(df)))
display(df.head(3))
display(df.target.value_counts().sort_index().rename(index=TARGET))

# 🔎 3. Kontrola danych
Brak tekstu, pusty tekst i duplikat to różne problemy. Nie usuwamy ich automatycznie: zmieniłoby to jednostkę analizy i liczebności grup.

### ▶️ Krok kodu 5

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
audit = {"n_documents":len(df), "missing_text":int(df.document.isna().sum()),
         "blank_text":int(df.document.fillna("").str.strip().eq("").sum()),
         "duplicate_occurrences":int(df.document.duplicated().sum()),
         "unique_texts":int(df.document.nunique())}
display(pd.Series(audit))
if audit["missing_text"] or audit["blank_text"]:
    raise ValueError("Najpierw uzgodnij sposób obsługi pustych/brakujących tekstów.")
duplicate_examples = df[df.document.duplicated(False)].head(4).copy()
duplicate_examples["document"] = duplicate_examples.document.str.slice(0, 240)
display(duplicate_examples)

## 🔎 3.1 Długość zależy od definicji
Znaki, elementy po spacji i tokeny słowne dają inne wyniki. Nasz jawny tokenizer zachowuje apostrofy wewnątrz słowa; nie jest pełnym modelem języka.

### ▶️ Krok kodu 6

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
WORD_PATTERN = r"[A-Za-z]+(?:'[A-Za-z]+)?"
df["n_chars"] = df.document.str.len()
df["n_whitespace"] = df.document.str.split().map(len)
df["raw_tokens"] = df.document.map(lambda t: re.findall(WORD_PATTERN, t))
df["n_words_raw"] = df.raw_tokens.map(len)
length_summary = df[["n_chars", "n_whitespace", "n_words_raw"]].describe()
display(length_summary.round(1))

# 🔎 4. Analiza
## 🔎 4.1 Małe litery, interpunkcja i sens
Lowercasing scala formy, ale usuwa wyróżnienie WIELKIMI LITERAMI. Usunięcie interpunkcji może utrudnić analizę negacji i sentymentu. To demonstracja, a nie zalecenie dla każdej metody.

### ▶️ Krok kodu 7

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
sample_index = df.document.str.contains(r"[!?]", regex=True).idxmax()
raw = df.loc[sample_index, "document"][:350]
before_after = pd.DataFrame({"wariant":["oryginał", "małe litery", "bez interpunkcji"],
                            "tekst":[raw, raw.lower(), re.sub(r"[^\w\s]", "", raw)]})
display(before_after)

## 🔎 4.2 URL, nazwy użytkowników i odstępy
Sprawdzamy obecność artefaktów w tym korpusie. Usuwanie odnośników może usunąć informację istotną dla pytania o ankiety lub autopromocję.

### ▶️ Krok kodu 8

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
URL = r"https?://\S+|www\.\S+"
USER = r"(?<!\w)(?:u/|@)[A-Za-z0-9_-]+"
artifact_counts = pd.Series({"URL":df.document.str.contains(URL, regex=True).sum(),
                             "użytkownik":df.document.str.contains(USER, regex=True).sum(),
                             "wielokrotne odstępy":df.document.str.contains(r"\s{2,}", regex=True).sum()})
display(artifact_counts)
def minimal_clean(text):
    text = re.sub(URL, " ", text)
    text = re.sub(USER, " ", text)
    return re.sub(r"\s+", " ", text).strip()
df["document_clean"] = df.document.map(minimal_clean)
chosen = df[df.document.str.contains(URL, regex=True)].head(2)
display(chosen[["document", "document_clean"]].map(lambda x: x[:300]))

## 🔎 4.3 Tokenizacja
Dla słowników zachowujemy stopwords i negację w mianowniku. Zapisujemy tokeny po łagodnym czyszczeniu i lowercasingu.

### ▶️ Krok kodu 9

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
df["tokens"] = df.document_clean.map(lambda t: re.findall(WORD_PATTERN, t.lower()))
df["n_words"] = df.tokens.map(len)
print("Tekst:", raw)
print("Tokeny:", re.findall(WORD_PATTERN, minimal_clean(raw).lower()))
print("Wielkość słownika korpusu:", len(set(t for tokens in df.tokens for t in tokens)))
print("Dokumenty bez tokenów:", int(df.n_words.eq(0).sum()))

## 🔎 4.4 Stopwords, stemming i lematyzacja
Stem jest mechanicznym skrótem, lemma formą podstawową wyznaczaną z kontekstu. spaCy też może się mylić. **Nie** podmieniamy oryginałów na te formy w analizie VADER ani przy embeddingach.

### ▶️ Krok kodu 10

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
try:
    import spacy
    from nltk.stem import PorterStemmer
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except (ImportError, OSError) as exc:
    raise RuntimeError("Zainstaluj ../requirements_workshop.txt, w tym model en_core_web_sm.") from exc
stemmer = PorterStemmer()
parsed = nlp(raw)
display(pd.DataFrame([{"token":t.text, "stopword":t.is_stop, "stem":stemmer.stem(t.text), "lemma":t.lemma_}
                      for t in parsed if t.is_alpha]).head(20))
print("Bez stopwords:", [t.text for t in parsed if t.is_alpha and not t.is_stop])

## 🔎 4.5 Przetworzenie korpusu
Zachowujemy dwa warianty: lemmy wszystkich słów do demonstracji dopasowania słownika oraz lemmy bez stopwords do porównania częstości. Oryginał pozostaje osobną kolumną.

### ▶️ Krok kodu 11

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
lemmas, content_lemmas = [], []
for doc in nlp.pipe(df.document_clean.tolist(), batch_size=64):
    lemmas.append([t.lemma_.lower() for t in doc if t.is_alpha])
    content_lemmas.append([t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop])
df["tokens_lemma"] = lemmas
df["tokens_content"] = content_lemmas
df["document_lemma"] = df.tokens_content.map(" ".join)
display(df[["document", "document_lemma"]].head(2).map(lambda x: x[:350]))

## 🔎 4.6 Unigramy, bigramy, trigramy
N-gram zachowuje lokalną kolejność tokenów. Nie przekraczamy granicy dokumentu. Usunięcie stopwords wcześniej może tworzyć sztuczne sąsiedztwa.

### ▶️ Krok kodu 12

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
def ngrams(tokens, n):
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
for n in [1, 2, 3]:
    print(n, "gram:", ngrams(df.tokens.iloc[0], n)[:10])
bigram_counts = Counter(gram for tokens in df.tokens for gram in ngrams(tokens, 2))
display(pd.DataFrame(bigram_counts.most_common(10), columns=["bigram", "count"]))

# 🔎 5. Wykresy
Długi prawy ogon uzasadnia pokazanie mediany obok średniej. Oś liczby dokumentów jest logarytmiczna.

### ▶️ Krok kodu 13

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** wykres.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df.n_words_raw, bins=50, color="#2B8C8E")
ax.set(xlabel="Liczba tokenów słownych", ylabel="Liczba dokumentów (log)", yscale="log", title="Długość dostarczonych tekstów")
fig.tight_layout(); fig.savefig(OUT / "length_histogram.png"); plt.show()

## 🔎 5.1 Najczęstsze tokeny przed i po
Porównanie pokazuje zmianę reprezentacji, a nie odkrycie, że pominięte słowa są bez znaczenia.

### ▶️ Krok kodu 14

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** wykres.

In [ ]:
before = Counter(t.lower() for tokens in df.raw_tokens for t in tokens)
after = Counter(t for tokens in df.tokens_content for t in tokens)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, counts, title in zip(axes, [before, after], ["Przed: tokeny", "Po: lemmy bez stopwords"]):
    terms, values = zip(*counts.most_common(10)[::-1])
    ax.barh(terms, values, color="#2B8C8E"); ax.set_title(title); ax.set_xlabel("Wystąpienia")
fig.tight_layout(); fig.savefig(OUT / "tokens_before_after.png"); plt.show()

# 🔎 6. Interpretacja
Jedna lista operacji nie pasuje do wszystkich metod. Słownik wymaga zgodności tokenizacji i swoich haseł. VADER wykorzystuje negację, interpunkcję i wielkie litery. Transformer powinien dostać zrozumiałe zdania; agresywna lematyzacja odbiera mu kontekst. Zbiór zawiera powtórzenia; późniejsze przedziały i testy będą warunkowe względem założenia niezależności dokumentów.

## 🔎 Zapis wspólnej ramki
Listy kodujemy jako JSON; następny notebook użyje `json.loads`, nigdy `eval`.

VADER może wykorzystywać emotikony. Słownik wymaga dopasowanej tokenizacji i lematyzacji haseł oraz tekstów. Transformer zwykle potrzebuje lżejszego przygotowania niż bag-of-words. Usuwanie stopwords zależy od pytania.

### ▶️ Krok kodu 15

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
export = df[["source_row", "target", "document", "document_clean", "tokens", "n_words", "tokens_lemma", "document_lemma"]].copy()
for column in ["tokens", "tokens_lemma"]:
    export[column] = export[column].map(lambda x: json.dumps(x, ensure_ascii=False))
export.to_csv(ROOT / "data/workshop_preprocessed.csv", index=False)
length_summary.to_csv(OUT / "length_summary.csv")
audit.update({"mean_words":float(df.n_words_raw.mean()), "median_words":float(df.n_words_raw.median()),
              "runtime_seconds":time.perf_counter()-START})
(OUT / "summary.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
before_after.to_json(OUT / "before_after.json", orient="records", force_ascii=False)
print("Zapisano", len(export), "wierszy do data/workshop_preprocessed.csv")

# 🔎 7. Ćwiczenie
1. Wybierz dokument z negacją i porównaj wariant z/bez stopwords. Co zmieniło się w sensie?
2. Zmień definicję tokena: czy liczba słów i mianownik przyszłego wskaźnika są takie same?
3. Czy usunąłbyś duplikaty w badaniu częstości postów, a czy w badaniu różnych treści? Uzasadnij przed analizą.

**Przejście:** w module 2 nadamy wybranym tokenom znaczenie za pomocą jawnych kategorii badacza.

## 💾 Pobranie wyników

Uruchom tę komórkę po ukończeniu analizy. Utworzy ZIP i w Colab rozpocznie pobieranie. Jeśli przeglądarka je zablokuje, odszukaj ZIP w panelu Pliki i pobierz ręcznie. Zachowaj też własną kopię notebooka.

In [ ]:
import shutil
bundle = Path('wyniki_modul_01')
bundle.mkdir(exist_ok=True)
if not Path(OUT).exists():
    raise RuntimeError('⛔ Najpierw wykonaj komórki analizy i zapisu wyników.')
shutil.copytree(OUT, bundle / 'tabele_i_wykresy', dirs_exist_ok=True)
shutil.copy2(ROOT / 'data/workshop_preprocessed.csv', bundle / 'workshop_preprocessed.csv')
archive = shutil.make_archive(str(bundle), 'zip', bundle)
print('✅ Plik wyników:', archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)
